### Volume / Row Count Check

In [ ]:
import os
import glob
import pandas as pd
# 1. Khai báo đường dẫn thư mục chứa dataset
folder_path = rf'D:\olist-customer-satisfaction-analysis\data\raw'

# 2. Lấy danh sách tất cả các file .csv trong thư mục
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

total_lines = 0

print(f"Đang đếm số dòng cho {len(csv_files)} files...\n")

for file in csv_files:
    # Mở file và đếm số dòng (dùng generator để không tốn RAM)
    with open(file, 'r', encoding='utf-8', errors='ignore') as f:
        # Trừ 1 để không tính dòng Header (tên cột)
        lines = sum(1 for _ in f) - 1 
        
    total_lines += lines
    print(f"{os.path.basename(file)}: {lines:,} dòng")

print("-" * 30)
print(f"=> Tổng cộng: {total_lines:,} dòng dữ liệu")

Đang đếm số dòng cho 9 files...

olist_customers_dataset.csv: 99,441 dòng
olist_geolocation_dataset.csv: 1,000,163 dòng
olist_orders_dataset.csv: 99,441 dòng
olist_order_items_dataset.csv: 112,650 dòng
olist_order_payments_dataset.csv: 103,886 dòng
olist_order_reviews_dataset.csv: 104,719 dòng
olist_products_dataset.csv: 32,951 dòng
olist_sellers_dataset.csv: 3,095 dòng
product_category_name_translation.csv: 71 dòng
------------------------------
=> Tổng cộng: 1,556,417 dòng dữ liệu


In [ ]:
# ??m s? l??ng missing value ? t?ng c?t c?a m?i b?ng
from pathlib import Path
import pandas as pd

# T?m th? m?c data/raw t? th? m?c ch?a notebook, kh?ng ph? thu?c working directory
folder_path = Path("../data/raw")
if not folder_path.exists():
    folder_path = Path("data/raw")
csv_files = sorted(folder_path.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Kh?ng t?m th?y file CSV trong: {folder_path.resolve()}")

print(f"Ki?m tra missing value trong {len(csv_files)} b?ng...\n")

for file in csv_files:
    df = pd.read_csv(file)
    missing_by_column = df.isna().sum()
    missing_by_column = missing_by_column[missing_by_column > 0]

    print(f"{file.name} ({len(df):,} d?ng):")
    if missing_by_column.empty:
        print("  Kh?ng c? missing value")
    else:
        for column, count in missing_by_column.items():
            print(f"  {column}: {count:,} missing")
    print()

Nhận xét: Tất cả các table đều được import data đầy đủ. Riêng olist_order_reviews có sự khác biệt (104,719/99224). Nghi ngờ là do nội dung review có chứa xuống dòng

In [ ]:
path = r'D:\olist-customer-satisfaction-analysis\data\raw\olist_order_reviews_dataset.csv'

# Đọc không chia chunk, dùng python engine, bắt lỗi rõ ràng
df_full = pd.read_csv(
    path, 
    engine='python', 
    on_bad_lines='warn',   # sẽ in ra dòng nào bị lỗi parse
    encoding='utf-8'
)

print(f"Tổng số dòng đọc được: {len(df_full)}")

# Đếm số dòng thực tế trong file CSV gốc (kể cả header)
with open(path, 'r', encoding='utf-8', errors='replace') as f:
    raw_line_count = sum(1 for _ in f)
print(f"Số dòng raw trong file (ước lượng, có thể lệch do multiline field): {raw_line_count - 1}")

Tổng số dòng đọc được: 99224
Số dòng raw trong file (ước lượng, có thể lệch do multiline field): 104719


### Missing Value Check per Table
??m gi? tr? thi?u theo t?ng c?t trong t?t c? CSV ? `data/raw`. Ch? hi?n th? c?c c?t c? ?t nh?t m?t missing value.

In [ ]:
from pathlib import Path

folder_path = Path('..') / 'data' / 'raw'
csv_files = sorted(folder_path.glob('*.csv'))

missing_summaries = []
for file in csv_files:
    df = pd.read_csv(file)
    missing_counts = df.isna().sum()
    missing_counts = missing_counts[missing_counts > 0]

    if missing_counts.empty:
        print(f'{file.name}: kh?ng c? missing value ({len(df):,} d?ng)')
    else:
        table = missing_counts.rename('missing_count').to_frame()
        table['missing_pct'] = (table['missing_count'] / len(df) * 100).round(2)
        table.index.name = 'column'
        print(f'\n{file.name} ({len(df):,} d?ng)')
        display(table)
        missing_summaries.append((file.name, table))

print(f'\n?? ki?m tra {len(csv_files)} b?ng.')